# Research Playbook

**The canonical path from raw data to backend registration.**

Every cell calls an existing function from `src/research/`. No inline logic, no black boxes.
Agent automation runs the same path — this notebook is both the research sandbox and the deployment contract.

| Step | What happens | Module called |
|------|-------------|---------------|
| 1 | Config & run_id | — |
| 2 | Data preparation | `pipeline.py` |
| 3 | Label construction | `cross_sectional_training.py` |
| 4 | Model training | `cross_sectional_training.py` |
| 5 | Walk-forward validation | `walk_forward.py` |
| 6 | Backtest (TopN Spread) | `backtest_core.py` |
| 7 | Register to backend | `workflow_store.py` |
| 8 | Inspect stored result | `workflow_store.py` |


## Step 1 — Config

In [ ]:
import uuid
from pathlib import Path

# ── Editable config ────────────────────────────────────────────────────────
MARKET        = "us"          # 'us' | 'cn'
MODEL_TYPE    = "lgbm"        # model family
TOPK          = 10             # long / short leg size
HOLDING_DAYS  = 10             # rebalance frequency
TRAIN_START   = "2020-01-01"
TRAIN_END     = "2024-12-31"
BACKTEST_START= "2025-01-01"
BACKTEST_END  = "2026-06-30"
BENCHMARK     = "QQQ"         # benchmark ticker
GOAL          = "TopN alpha — winner-bucket LGBMClassifier on Alpha158"

# Deterministic run_id so reruns are idempotent
RUN_ID = f"{MARKET}_{MODEL_TYPE}_{TRAIN_END.replace('-','')}_{uuid.uuid4().hex[:8]}"
print("run_id:", RUN_ID)

## Step 2 — Data Preparation

In [ ]:
# Initialise qlib and load raw features via the canonical pipeline.
# pipeline.py owns all qlib init logic — do not call qlib.init() directly here.
from src.research.pipeline import ResearchPipeline

pipeline = ResearchPipeline(market=MARKET)
dataset = pipeline.load_dataset(
    start_time=TRAIN_START,
    end_time=BACKTEST_END,
)
print("Dataset loaded:", type(dataset))

# ── Sanity check ────────────────────────────────────────────────────────────
# Inspect feature shape and date coverage before spending compute on training.
train_df = pipeline.get_feature_dataframe(dataset, start=TRAIN_START, end=TRAIN_END)
print(f"Train feature shape: {train_df.shape}")
print(f"Date range: {train_df.index.get_level_values('datetime').min()} →
      {train_df.index.get_level_values('datetime').max()}")
train_df.head()

## Step 3 — Label Construction

In [ ]:
# Build winner-bucket labels: 1 if stock is in top-N by forward return, else 0.
# Calls the same label logic used in production training.
from src.research.cross_sectional_training import CrossSectionalTrainer

trainer = CrossSectionalTrainer(market=MARKET, topk=TOPK, holding_days=HOLDING_DAYS)
X_train, y_train, label_meta = trainer.build_labels(
    dataset=dataset,
    start=TRAIN_START,
    end=TRAIN_END,
)
print(f"X_train: {X_train.shape}  |  y_train positive rate: {y_train.mean():.3f}")
print("Label meta:", label_meta)

# ── Sanity check ────────────────────────────────────────────────────────────
# Positive rate should be close to topk / universe_size.
# If it is 0 or 1, the label construction has a bug.
assert 0.02 < y_train.mean() < 0.5, f"Unexpected positive rate: {y_train.mean():.3f}"

## Step 4 — Model Training

In [ ]:
# Train the LGBMClassifier and get winner_prob scores.
# trainer.fit() returns the fitted model and OOF predictions for inspection.
model, oof_preds, train_metrics = trainer.fit(X_train, y_train)

print("Train metrics:")
for k, v in train_metrics.items():
    print(f"  {k}: {v:.4f}")

# ── Sanity check ────────────────────────────────────────────────────────────
# AUC should be comfortably above 0.5.
# If AUC ~ 0.5, the model has no predictive power — stop here and revisit features.
assert train_metrics.get("auc", 0) > 0.52, "AUC too low — revisit features or label horizon"

## Step 5 — Walk-Forward Validation

In [ ]:
# Walk-forward ensures no look-ahead bias.
# Uses the same rolling-window logic as production via walk_forward.py.
from src.research.walk_forward import WalkForwardValidator

wfv = WalkForwardValidator(
    market=MARKET,
    topk=TOPK,
    holding_days=HOLDING_DAYS,
)
wf_result = wfv.run(
    dataset=dataset,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_end=BACKTEST_END,
)

print("Walk-forward summary:")
print(f"  Mean fold AUC : {wf_result.mean_auc:.4f}")
print(f"  Mean fold IC  : {wf_result.mean_ic:.4f}")
print(f"  Stable folds  : {wf_result.stable_folds} / {wf_result.total_folds}")

# ── Stop condition ──────────────────────────────────────────────────────────
# If fewer than half the folds are stable, the model is not deployable.
assert wf_result.stable_folds / wf_result.total_folds >= 0.5, \
    f"Only {wf_result.stable_folds}/{wf_result.total_folds} stable folds — not deployable"

## Step 6 — Backtest (TopN Spread)

In [ ]:
# Use backtest_core pure functions — same logic as backtest_strategy.py
# but callable without fire.Fire() or CLI args.
import pandas as pd
import matplotlib.pyplot as plt
from src.research.backtest_core import (
    select_topn_with_guardrail,
    select_topn_no_guardrail,
    compute_portfolio_returns,
    compute_turnover,
    build_backtest_summary,
)

# ── Generate out-of-sample scores ──────────────────────────────────────────
oos_df = pipeline.get_feature_dataframe(dataset, start=BACKTEST_START, end=BACKTEST_END)
scores_by_date = trainer.predict_by_date(model, oos_df)   # dict[date, pd.Series]

# ── Load price data for guardrail ───────────────────────────────────────────
close_prices, ma60 = pipeline.load_prices_and_ma(
    start=BACKTEST_START, end=BACKTEST_END, ma_window=60
)

# ── Build holdings ──────────────────────────────────────────────────────────
holdings_long, holdings_short = {}, {}
for date, scores in scores_by_date.items():
    px = close_prices.get(date, {})
    ma = ma60.get(date, {})
    holdings_long[date]  = select_topn_with_guardrail(scores, TOPK, px, ma)
    holdings_short[date] = select_topn_no_guardrail(scores, TOPK)

# ── Compute returns ─────────────────────────────────────────────────────────
daily_rets = pipeline.load_daily_returns(start=BACKTEST_START, end=BACKTEST_END)
bench_ret  = pipeline.load_benchmark_returns(BENCHMARK, start=BACKTEST_START, end=BACKTEST_END)

long_ret  = compute_portfolio_returns(holdings_long,  daily_rets)
short_ret = compute_portfolio_returns(holdings_short, daily_rets)
spread    = long_ret - short_ret

# ── Turnover ────────────────────────────────────────────────────────────────
avg_turnover = compute_turnover(holdings_long).mean()

# ── Summary dataclass ───────────────────────────────────────────────────────
summary = build_backtest_summary(
    long_returns=long_ret,
    short_returns=short_ret,
    bench_returns=bench_ret,
    avg_turnover=avg_turnover,
)

print(f"Spread (annualised) : {summary.annualised_spread:.2%}")
print(f"Alpha (long vs bench): {summary.alpha_long:.2%}")
print(f"Max drawdown        : {summary.max_drawdown:.2%}")
print(f"Avg turnover/rebal  : {avg_turnover:.2%}")

In [ ]:
# ── Visualisation ───────────────────────────────────────────────────────────
from pathlib import Path
Path('artifacts').mkdir(exist_ok=True)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

(1 + long_ret).cumprod().rename('Long').plot(ax=axes[0], color='steelblue')
(1 + short_ret).cumprod().rename('Short').plot(ax=axes[0], color='tomato')
(1 + bench_ret).cumprod().rename(BENCHMARK).plot(ax=axes[0], color='gray', linestyle='--')
axes[0].set_title('Cumulative Return'); axes[0].legend()

(1 + spread).cumprod().rename('Long-Short Spread').plot(ax=axes[1], color='purple')
axes[1].axhline(1, color='black', linewidth=0.5)
axes[1].set_title('Spread (Long − Short)')

excess = long_ret - bench_ret
excess.cumsum().rename('Cumulative Alpha').plot(ax=axes[2], color='darkgreen')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('Cumulative Alpha (Long − Benchmark)')

plt.tight_layout()
plt.savefig(f'artifacts/{RUN_ID}_backtest.png', dpi=150)
plt.show()

## Step 7 — Register to Backend

In [ ]:
# Persist the run as a ResearchWorkflowResult so the backend can discover it.
# This is the ONLY write operation in the notebook — everything above is read-only.
from src.research.workflow_store import ResearchWorkflowStore
from src.research.workflow_types import (
    ResearchWorkflowRequest,
    ResearchWorkflowResult,
    ResearchStep,
    StepResult,
    WorkflowStatus,
    utc_now,
)

steps = [
    StepResult(step=ResearchStep.SCAN,     status=WorkflowStatus.COMPLETED,
               output={'feature_set': 'Alpha158', 'market': MARKET}),
    StepResult(step=ResearchStep.COMPILE,  status=WorkflowStatus.COMPLETED,
               output={'train_rows': int(X_train.shape[0]),
                        'positive_rate': float(y_train.mean())}),
    StepResult(step=ResearchStep.TRAIN,    status=WorkflowStatus.COMPLETED,
               output=train_metrics),
    StepResult(step=ResearchStep.WALK_FORWARD, status=WorkflowStatus.COMPLETED,
               output={'mean_auc': float(wf_result.mean_auc),
                        'mean_ic': float(wf_result.mean_ic),
                        'stable_folds': wf_result.stable_folds,
                        'total_folds': wf_result.total_folds}),
    StepResult(step=ResearchStep.BACKTEST, status=WorkflowStatus.COMPLETED,
               output={'annualised_spread': float(summary.annualised_spread),
                        'alpha_long': float(summary.alpha_long),
                        'max_drawdown': float(summary.max_drawdown),
                        'avg_turnover': float(avg_turnover),
                        'topk': TOPK,
                        'holding_days': HOLDING_DAYS,
                        'benchmark': BENCHMARK}),
    StepResult(step=ResearchStep.ATTRIBUTION, status=WorkflowStatus.SKIPPED),
    StepResult(step=ResearchStep.PROMOTE,     status=WorkflowStatus.PENDING),
]

request = ResearchWorkflowRequest(
    market=MARKET,
    goal=GOAL,
    model_type=MODEL_TYPE,
    run_id=RUN_ID,
    requested_by='notebook',
    metadata={'topk': TOPK, 'holding_days': HOLDING_DAYS,
              'train_start': TRAIN_START, 'train_end': TRAIN_END,
              'backtest_start': BACKTEST_START, 'backtest_end': BACKTEST_END},
)
result = ResearchWorkflowResult(
    run_id=RUN_ID,
    request=request,
    status=WorkflowStatus.COMPLETED,
    steps=steps,
    started_at=utc_now(),
    completed_at=utc_now(),
)

store = ResearchWorkflowStore()
saved_path = store.save(result)
print(f'Saved to: {saved_path}')
print(f'run_id  : {RUN_ID}')

## Step 8 — Inspect Stored Result

In [ ]:
# Reload the result from disk and show all step outputs.
# This is exactly what the backend adapter reads when it polls for new runs.
loaded = store.load(RUN_ID)

print(f'run_id : {loaded.run_id}')
print(f'status : {loaded.status}')
print(f'market : {loaded.request.market}')
print(f'goal   : {loaded.request.goal}')
print()

for step in loaded.steps:
    icon = {'completed': '✅', 'skipped': '⏭',
            'pending': '⏳', 'failed': '❌'}.get(step.status.value, '?')
    print(f'{icon}  {step.step.value:<15} {step.status.value}')
    if step.output:
        for k, v in step.output.items():
            print(f'       {k}: {v}')

print('\n─── Recent runs ───')
for run in store.list(limit=5):
    print(f"  {run['run_id']}  status={run['status']}  completed={run.get('completed_at','?')[:10]}")

---
## Decision table — what to do after this notebook

| Condition | Action |
|-----------|--------|
| `annualised_spread > 0.15` AND `stable_folds >= total_folds * 0.7` | **Promote**: update `ResearchStep.PROMOTE` to COMPLETED → backend picks up the run and triggers `inference.py` |
| Spread reasonable but `max_drawdown > 0.30` | Revisit guardrail params in Step 6 (score threshold, MA window) |
| AUC stuck near 0.5 | Go back to Step 3 — try longer label horizon or sector-neutral labels |
| Walk-forward unstable | Increase train window or reduce feature count |
